# Prepping Data Challenge - Week 18 

## Import Libraries

In [1]:
import numpy as np
import pandas as pd

## Stage 1

### Import Data

In [2]:
wildlife_df = pd.read_excel('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_18_Zoo_Data.xlsx', sheet_name='Wildlife')
animal_details_df = pd.read_excel('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_18_Zoo_Data.xlsx', sheet_name='Animal Details')
plant_details_df = pd.read_excel('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_18_Zoo_Data.xlsx', sheet_name='Plant Details')

In [3]:
wildlife_df.head()

,Wildlife,Name,Region,Habitat
0,Animal,African Penguin,Africa,Oceans and Coastline
1,Animal,African Pygmy Falcon,Africa,Savanna
2,Animal,African Pygmy Goose,Africa,Rivers and Lakes
3,Animal,African Spurred Tortoise,Africa,"Desert, Savanna"
4,Animal,Agouti,"Central America, South America",Tropical Rainforest


In [4]:
animal_details_df.head()

,Animal,Status,Class,Family,Genus,Order,Species
0,African Penguin,Endangered,Aves (Birds),Spheniscidae,Spheniscus,Sphenisciformes,demersus
1,African Pygmy Falcon,Stable,Aves (Birds),Falconidae,Polihierax,Falconiformes,semitorquatus
2,African Pygmy Goose,Stable,Aves (Birds),Anatidae,Nettapus,Anseriformes,auritus
3,African Spurred Tortoise,Threatened,Reptilla (Reptiles),Testudinidae,Geochelone,Testudines,sulcata
4,Agouti,Some Endangered,Mammalia (Mammals),Dasyprocidae,Dasyprocta,Rodentia,11


In [5]:
plant_details_df.head()

,Plant,Stattus,Class,Division,Family,Genus,Order,Species
0,Acacia,Some Endangered,Magnoliopsida,Tracheophyta - vascular plants,Fabaceae - legumes,"Acacia, Acaciella, Mariosousa, Senegalia and V...",Fabales,"more than 1,400"
1,African Protea,Some Endangered,Equisetopsida,Magnoliophyta,Proteaceae,"14 genera, including Protea and Leucospermum",Proteales,about 360 species
2,African Tulip Tree,Stable,Magnoliopsida,Tracheophyta - vascular plants,Bignoniaceae,Spathodea,Scrophulariales,campanulata
3,Agave,Some Endangered,Liliopsida,Magnoliophyta - flowering plants,Agavaceae - Agave Family,Agave,Asparagales,More than 200
4,Aloe,Some Endangered,Liliopsida,Tracheophyta,Asphodelaceae,Aloe,Liliales,more than 450


### Combine animal and plant into one table

The easiest way to do this is to concatenate. For this, the names of the columns must match. For our analysis we will also only need the `Name`, `Table Name`, `Status`, and `Class` columns.

In [6]:
# Rename columns to ensure consistency
plant_details_df = plant_details_df.rename(columns={'Plant' : 'Name', 'Stattus': 'Status'})
animal_details_df = animal_details_df.rename(columns={'Animal' : 'Name'})

# Add a Table Name (Plant/Animal) column. Call it 'Wildlife' for better merging later on
animal_details_df['Wildlife'] = 'Animal'
plant_details_df['Wildlife'] = 'Plant'

# Keep only a limited number of columns required for analysis
ANALYSIS_COLUMNS = ['Name', 'Wildlife', 'Status', 'Class']

# Combine Plants and Animals
wildlife_details_df = pd.concat([animal_details_df[ANALYSIS_COLUMNS].copy(), plant_details_df[ANALYSIS_COLUMNS].copy()]).reset_index(drop=True)

### Only use English class name where possible

In [7]:
def keep_english_name(name):
    
    # Convert to string
    name = str(name)

    # Check if name is actually empty and keep empty data type
    if name == 'nan':
        return np.nan
    
    # Check if there is an English name (in parentheses)
    if '(' in name:
        english_name = name.split(' ')[1] # The english name is listed as the second element
        english_name = english_name.replace('(', '').replace(')','') # Remove the parentheses
        return english_name
    # If no English name exists, return the standard name
    else:
        return name

In [8]:
wildlife_details_df['Class'] = wildlife_details_df['Class'].apply(keep_english_name)

In [9]:
wildlife_details_df['Class'].unique()

array(['Birds', 'Reptiles', 'Mammals', 'Insects', 'Amphibians',
       'Ray-finned', 'Millipedes', nan, 'Chondrichthyes', 'Arachnids',
       'Sarcopterygii', 'Magnoliopsida', 'Equisetopsida', 'Liliopsida',
       'Pteridopsida', 'Conifers', 'Cycadopsida', 'Polypodiospida',
       'Asterids'], dtype=object)

### Combine Wildlife DF and Wildlife Details DF

In [10]:
# Fix wildlife Names
wildlife_df['Name'] = wildlife_df['Name'].apply(lambda x: str(x).replace('&#039;', "'"))

# Join wildlife Data
wildlife_df = wildlife_df.merge(wildlife_details_df, on=['Name', 'Wildlife'])

In [11]:
wildlife_df.head()

,Wildlife,Name,Region,Habitat,Status,Class
0,Animal,African Penguin,Africa,Oceans and Coastline,Endangered,Birds
1,Animal,African Pygmy Falcon,Africa,Savanna,Stable,Birds
2,Animal,African Pygmy Goose,Africa,Rivers and Lakes,Stable,Birds
3,Animal,African Spurred Tortoise,Africa,"Desert, Savanna",Threatened,Reptiles
4,Animal,Agouti,"Central America, South America",Tropical Rainforest,Some Endangered,Mammals


In [12]:
wildlife_df.shape

(326, 6)

## Stage 2

### Add additional Data Sources

In [13]:
climate_df = pd.read_excel('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_18_Zoo_Data.xlsx', sheet_name='San Diego Climate')
habitats_df = pd.read_excel('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_18_Zoo_Data.xlsx', sheet_name='Habitats (estimates)')
care_priority_df = pd.read_excel('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_18_Zoo_Data.xlsx', sheet_name='Care Priority')

In [14]:
climate_df

,High °F,Low °F,Month,High °C,Low °C
0,66,50,January,19,10
1,66,52,February,19,11
2,67,55,March,19,13
3,69,57,April,20,14
4,70,60,May,21,16
5,72,63,June,22,17
6,75,66,July,24,19
7,77,68,August,25,20
8,77,66,September,25,19
9,75,62,October,24,16


In [15]:
habitats_df

,Habitat,Min Temp,Max Temp,Elevation,Min Rainfall,Max Rainfall
0,Desert,–4 degrees Fahrenheit (–20 degrees Celsius),120 degrees Fahrenheit (50 degrees Celsius),NaN,NaN,NaN
1,Island,69 degrees Fahrenheit (21 degrees Celsius),84 degrees Fahrenheit (30 degrees Celsius),NaN,NaN,NaN
2,Mountains,23 degrees Fahrenheit (-5 degrees Celsius),75 degrees Fahrenheit (24 degrees Celsius),"over 1,900 feet",NaN,NaN
3,Oceans and Coastline,28 degrees Fahrenheit (-2 degrees Celsius),86 degrees Fahrenheit (30 degrees Celsius),NaN,NaN,NaN
4,Prairie and Steppes,-4 degrees Fahrenheit (-20 degrees Celsius),86 degrees Fahrenheit (30 degrees Celsius),NaN,NaN,NaN
5,Rivers and Lakes,35 degrees Fahrenheit (2 degrees Celsius),75 degrees Fahrenheit (24 degrees Celsius),NaN,NaN,NaN
6,Savanna,50 degrees Fahrenheit (10 degrees Celsius),86 degrees Fahrenheit (30 degrees Celsius),NaN,NaN,NaN
7,Scrubland,30 degrees Fahrenheit (-1 degrees Celsius),100 degrees Fahrenheit (38 degrees Celsius),NaN,NaN,NaN
8,Temperate Forest and Taiga,-4 degrees Fahrenheit (-20 degrees Celsius),64 degrees Fahrenheit (18 degrees Celsius),NaN,NaN,NaN
9,Tropical Rainforest,68 degrees Fahrenheit (20 degrees Celsius),82 degrees Fahrenheit (28 degrees Celsius),NaN,60 inches (1.5 meters) per year,200 inches (5 meters) per year


In [16]:
care_priority_df

,Priority,Status
0,1,Endangered
1,2,Some Endangered
2,3,Threatened
3,4,Some Threatened
4,5,Stable


### Simplify climate DataFrame

In [17]:
climate_range_df = pd.DataFrame({'Low °C' : climate_df['Low °C'].min(), 'High °C' : climate_df['High °C'].max()}, index=[0])

In [18]:
climate_range_df

,Low °C,High °C
0,10,25


### Add Climate Information based on Habitat

In [19]:
# Split up the multiple habitat column into multiple rows
wildlife_df['Habitat'] = wildlife_df['Habitat'].apply(lambda x: str(x).split(','))

wildlife_df = wildlife_df.explode('Habitat')

In [20]:
wildlife_df

,Wildlife,Name,Region,Habitat,Status,Class
0,Animal,African Penguin,Africa,Oceans and Coastline,Endangered,Birds
1,Animal,African Pygmy Falcon,Africa,Savanna,Stable,Birds
2,Animal,African Pygmy Goose,Africa,Rivers and Lakes,Stable,Birds
3,Animal,African Spurred Tortoise,Africa,Desert,Threatened,Reptiles
3,Animal,African Spurred Tortoise,Africa,Savanna,Threatened,Reptiles
...,...,...,...,...,...,...
323,Plant,Woolly Blue Curls,North America,Scrubland,Stable,Magnoliopsida
324,Plant,Yellow Bells,"Caribbean, Central America, North America, Sou...",Island,NaN,Magnoliopsida
324,Plant,Yellow Bells,"Caribbean, Central America, North America, Sou...",Scrubland,NaN,Magnoliopsida
325,Plant,Yucca,"Central America, North America",Desert,Some Threatened,Liliopsida


### Add habitat temperature range information to wildlife Data Frame

In [29]:
def extract_celsius(temperature_string):
    """Used to extract temperature information in celsius and cast it as an integer value from text column in Habitat DataFrame.
       Assumption: Temperature string has the Format: F degrees Fahrenheit (C degrees Celsius). This function extracts C as an integer Value.
        
        param temperature_string (str): temperature information as a string.

        returns temperature_in_celsius (int)
    """
    if str(temperature_string) == 'nan':
        return np.nan
        
    
    temperature_celsius_string = str(temperature_string).split(' (')[1]
    temperature_in_celsius = temperature_celsius_string.split(' ')[0]
    temperature_in_celsius = temperature_in_celsius.replace('–', '-')

    return int(temperature_in_celsius)

In [31]:
habitats_df['Min °C'] = habitats_df['Min Temp'].apply(extract_celsius)
habitats_df['Max °C'] = habitats_df['Max Temp'].apply(extract_celsius)

In [34]:
wildlife_df = wildlife_df.merge(habitats_df[['Habitat', 'Min °C', 'Max °C']], on='Habitat')

In [35]:
wildlife_df.head()

,Wildlife,Name,Region,Habitat,Status,Class,Min °C,Max °C
0,Animal,African Penguin,Africa,Oceans and Coastline,Endangered,Birds,-2,30
1,Animal,African Pygmy Falcon,Africa,Savanna,Stable,Birds,10,30
2,Animal,African Pygmy Goose,Africa,Rivers and Lakes,Stable,Birds,2,24
3,Animal,African Spurred Tortoise,Africa,Desert,Threatened,Reptiles,-20,50
4,Animal,Agouti,"Central America, South America",Tropical Rainforest,Some Endangered,Mammals,20,28


### Reduce the Table down to one temperature range per species

In [44]:
wildlife_df = wildlife_df.groupby(level=0).agg({'Wildlife' : 'first',
                                                'Name' : 'first',
                                                'Region' : 'first',
                                                'Habitat' : list,
                                                'Status' : 'first',   
                                                'Class' : 'first',
                                                'Min °C' : 'min', 
                                                'Max °C' : 'max', 
                                                })

In [45]:
wildlife_df.head()

,Wildlife,Name,Region,Habitat,Status,Class,Min °C,Max °C
0,Animal,African Penguin,Africa,[Oceans and Coastline],Endangered,Birds,-2,30
1,Animal,African Pygmy Falcon,Africa,[Savanna],Stable,Birds,10,30
2,Animal,African Pygmy Goose,Africa,[Rivers and Lakes],Stable,Birds,2,24
3,Animal,African Spurred Tortoise,Africa,[Desert],Threatened,Reptiles,-20,50
4,Animal,Agouti,"Central America, South America",[Tropical Rainforest],Some Endangered,Mammals,20,28


### Classify by whether or not the Animal is within it's natural climate

In [60]:
# Get sometimes hotter, colder, or whithin range
wildlife_df['Habitat Notes'] = np.where(wildlife_df['Max °C'] < climate_range_df['High °C'].loc[0], 'Above', np.nan)
wildlife_df['Habitat Notes'] = np.where(wildlife_df['Min °C'] > climate_range_df['Low °C'].loc[0], 'Below', wildlife_df['Habitat Notes'])
wildlife_df['Habitat Notes'] = np.where((wildlife_df['Max °C'] >= climate_range_df['High °C'].loc[0]) & (wildlife_df['Min °C'] <= climate_range_df['Low °C'].loc[0]), 'Ideal', wildlife_df['Habitat Notes'])

In [67]:
# Get degrees outside ideal temperature
wildlife_df['Degrees Outside Ideal'] = np.where(wildlife_df['Habitat Notes'] == 'Above', climate_range_df['High °C'].loc[0] - wildlife_df['Max °C'], np.nan)
wildlife_df['Degrees Outside Ideal'] = np.where(wildlife_df['Habitat Notes'] == 'Below', climate_range_df['Low °C'].loc[0] - wildlife_df['Min °C'], wildlife_df['Degrees Outside Ideal'])

### Filter by animals with non-ideal habitat

In [85]:
# Filter wildlife by suboptimal climate
non_ideal_climate_wildlife_df = wildlife_df[wildlife_df['Habitat Notes'].isin(['Above', 'Below'])]

# Add priority ranking
non_ideal_climate_wildlife_df = non_ideal_climate_wildlife_df.merge(care_priority_df, how='left')

# If no endangerment status, set priority to 6
non_ideal_climate_wildlife_df['Priority'] = non_ideal_climate_wildlife_df['Priority'].fillna(6)

In [89]:
non_ideal_climate_wildlife_df = non_ideal_climate_wildlife_df.sort_values(
    by=['Priority', 'Degrees Outside Ideal'],
    ascending=[True, False],
    key= lambda x : x.abs() if x.name == 'Degrees Outside Ideal' else x
)

In [90]:
non_ideal_climate_wildlife_df

,Wildlife,Name,Region,Habitat,Status,Class,Min °C,Max °C,Habitat Notes,Degrees Outside Ideal,Priority
32,Animal,Galápagos Tortoise,South America,[Island],Endangered,Reptiles,21,30,Below,-11.0,1.0
49,Animal,Komodo Dragon,Asia,[Island],Endangered,Reptiles,21,30,Below,-11.0,1.0
53,Animal,Lord Howe Island Stick Insect,Australia and New Zealand,[Island],Endangered,Insects,21,30,Below,-11.0,1.0
106,Animal,‘Alalā (Hawaiian crow),Pacific Islands,[Island],Endangered,Birds,21,30,Below,-11.0,1.0
109,Plant,"Alula, Cabbage on a Stick",Pacific Islands,[Island],Endangered,Magnoliopsida,21,30,Below,-11.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
115,Plant,Bonsai,"Africa, Asia, Australia and New Zealand, Carib...",[Temperate Forest and Taiga],None,None,-20,18,Above,7.0,6.0
129,Plant,Heavenly Bamboo,Asia,[Temperate Forest and Taiga],None,Magnoliopsida,-20,18,Above,7.0,6.0
135,Plant,Japanese Wisteria,Asia,[Temperate Forest and Taiga],None,Magnoliopsida,-20,18,Above,7.0,6.0
151,Plant,Sugarcane,Asia,[Temperate Forest and Taiga],None,Liliopsida,-20,18,Above,7.0,6.0


## Save Data

In [91]:
non_ideal_climate_wildlife_df.to_csv('D:01_Projekt_Portfolio/data_prepping_challenges/outputs/18_care_priority_list.csv')